<a href="https://colab.research.google.com/github/23wh1a1290/Agentic-AI/blob/main/DigitalTwinBotEdu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers faiss-cpu pymupdf

In [2]:
!pip install -q -U groq

In [3]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import fitz

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [4]:
def extract_pdf_text(pdf_path):

    doc = fitz.open(pdf_path)

    pages = []

    for page_number, page in enumerate(doc):

        text = page.get_text()

        if text.strip():
            pages.append({
                "page": page_number + 1,
                "text": text
            })

    return pages


def create_chunks(pages, chunk_size=180, overlap=40):

    chunks = []

    for page in pages:

        words = page["text"].split()

        start = 0

        while start < len(words):

            end = start + chunk_size

            chunk = " ".join(words[start:end])

            if chunk.strip():
                chunks.append({
                    "page": page["page"],
                    "text": chunk
                })

            start += chunk_size - overlap

    return chunks

In [5]:
vector_index = None
document_chunks = []


def process_pdf(pdf_path):

    global vector_index, document_chunks

    if not pdf_path:
        return "Please upload a PDF."

    pages = extract_pdf_text(pdf_path)

    if not pages:
        return "No readable text found in the PDF."

    document_chunks = create_chunks(pages)

    texts = [
        chunk["text"] for chunk in document_chunks
    ]

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    embeddings = embeddings.astype("float32")

    vector_index = faiss.IndexFlatIP(
        embeddings.shape[1]
    )

    vector_index.add(embeddings)

    return (
        f"PDF processed successfully!\n"
        f"Pages extracted: {len(pages)}\n"
        f"Chunks created: {len(document_chunks)}"
    )

In [6]:
def retrieve_context(query, top_k=4):

    if vector_index is None:
        return ""

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = vector_index.search(
        query_embedding,
        min(top_k, len(document_chunks))
    )

    results = []

    for idx in indices[0]:

        if idx >= 0:

            chunk = document_chunks[idx]

            results.append(
                f"Source Page {chunk['page']}:\n"
                f"{chunk['text']}"
            )

    return "\n\n".join(results)

In [7]:
from google.colab import userdata
from groq import Groq

groq_key = userdata.get("Groq")

client = Groq(api_key=groq_key)

MODEL_NAME = "openai/gpt-oss-20b"

print("Groq client initialized successfully!")

Groq client initialized successfully!


In [8]:
def educational_agent(topic, grade, difficulty, mode):

    if not topic or not topic.strip():
        return "Please enter a topic."

    mode = mode.strip().lower()

    # Retrieve information from uploaded PDF
    context = retrieve_context(
        topic,
        top_k=5
    )

    if context:
        reference = f"""
        Use the following retrieved information from the
        student's uploaded PDF as the primary reference.

        {context}

        Use only relevant information from this context.
        Do not invent textbook-specific facts.
        """
    else:
        reference = """
        No relevant information was retrieved from the PDF.
        Use your general knowledge and do not claim that
        the answer came from the uploaded textbook.
        """

    # MODE 1: LESSON
    if mode == "lesson":

        prompt = f"""
        You are an educational teacher.

        Generate a COMPLETE LESSON.

        Topic: {topic}
        Grade: {grade}
        Difficulty: {difficulty}

        REFERENCE MATERIAL:
        {reference}

        Follow ONLY this lesson structure:

        1. Lesson Title
        2. Learning Objectives
        3. Detailed Concept Explanation
        4. Important Concepts and Key Points
        5. Worked Examples
        6. Summary

        IMPORTANT:
        - Generate lesson notes only.
        - Explain concepts step by step.
        - Use grade-appropriate language.
        - Do not generate MCQ quizzes.
        - Do not generate a separate answer key.
        - Do not change the selected topic.

        """

    # MODE 2: QUIZ
    elif mode == "quiz":

        prompt = f"""
        You are an educational quiz generator.

        Generate ONLY A QUIZ.

        Topic: {topic}
        Grade: {grade}
        Difficulty: {difficulty}

        REFERENCE MATERIAL:
        {reference}

        STRICT OUTPUT FORMAT:

        # Quiz: {topic}

        Generate 10 multiple-choice questions.

        For every question provide:

        Question 1:
        Question text

        A) Option A
        B) Option B
        C) Option C
        D) Option D

        Repeat for all 10 questions.

        At the END provide:

        # Answer Key

        1. Correct option
        2. Correct option
        3. Correct option
        Continue for all questions.

        STRICT RULES:
        - Generate ONLY questions and their four options.
        - Provide ONLY the answer key at the end.
        - Do NOT generate learning objectives.
        - Do NOT generate an introduction.
        - Do NOT explain the topic.
        - Do NOT generate lesson notes.
        - Do NOT generate a summary.
        - Do NOT provide explanations for answers.
        - All questions must be based primarily on the
          retrieved reference material when available.
        - Do not ask questions unrelated to the topic.

        """

    # MODE 3: EXPLANATION
    elif mode == "explanation":

        prompt = f"""
        You are an educational subject expert.

        Generate ONLY A TOPIC EXPLANATION.

        Topic: {topic}
        Grade: {grade}
        Difficulty: {difficulty}

        REFERENCE MATERIAL:
        {reference}

        Follow this structure:

        # {topic}

        1. Concept Explanation
        Explain the topic clearly and step by step.

        2. Important Concepts
        Explain the important terms and ideas.

        3. Examples
        Give relevant examples to help students understand.

        STRICT RULES:
        - Explain only the selected topic.
        - Use simple, understandable language.
        - Do not generate MCQ questions.
        - Do not generate a quiz.
        - Do not generate learning objectives.
        - Do not generate lesson notes.
        - Do not generate an answer key.
        - Do not generate a lengthy summary.

        """

    else:
        return "Invalid content type selected."

    try:

        response = client.chat.completions.create(

            model=MODEL_NAME,

            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are an educational AI assistant. "
                        "Follow the selected content type strictly. "
                        "Never mix lesson, quiz and explanation formats."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],

            temperature=0.2,
            max_completion_tokens=2500,
            reasoning_effort="low"
        )

        result = response.choices[0].message.content

        return result if result else "No content generated."

    except Exception as e:

        return f"Generation Error: {str(e)}"

In [ ]:
import gradio as gr

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🎓 Educational AI Agent")
    gr.Markdown(
        "Generate personalized educational content "
        "using AI and uploaded textbooks."
    )

    gr.Markdown("## Upload Study Material")

    pdf_upload = gr.File(
        label="Upload PDF Textbook",
        file_types=[".pdf"],
        type="filepath"
    )

    process_btn = gr.Button("Process PDF 📚")

    pdf_status = gr.Textbox(
        label="PDF Processing Status"
    )

    process_btn.click(
        fn=process_pdf,
        inputs=pdf_upload,
        outputs=pdf_status
    )

    gr.Markdown("## Generate Educational Content")

    with gr.Row():

        topic = gr.Textbox(label="Topic")

        grade = gr.Dropdown(
            choices=[str(i) for i in range(1, 13)],
            value="12",
            label="Grade"
        )

    with gr.Row():

        difficulty = gr.Dropdown(
            ["Easy", "Medium", "Hard"],
            value="Medium",
            label="Difficulty Level"
        )

        mode = gr.Radio(
            ["Lesson", "Quiz", "Explanation"],
            value="Lesson",
            label="Content Type"
        )

    generate_btn = gr.Button(
        "Generate Content 🚀",
        variant="primary"
    )

    output = gr.Markdown()

    generate_btn.click(
        fn=educational_agent,
        inputs=[topic, grade, difficulty, mode],
        outputs=output
    )

demo.queue().launch(share=True, debug=True)

/tmp/ipykernel_18726/1356320778.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bce60fe0fc809a2c5d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
